# Pretraining: a companion notebook

**Human Language Technologies 2026-27, Lecture 17.** Claudio Gallicchio, University of Pisa.

This notebook lets you run the examples of the lecture, pretrain a (very) small transformer language model from scratch, and check the exercises of the exercise sheet. It follows Chapter 7 of Jurafsky and Martin, _Speech and Language Processing_ (3rd edition draft, August 2026), sections 7.7 and 7.8, and Chapter 1, sections 1.1, 1.6.1 and 1.9, with the scaling laws of Kaplan et al. (2020).

1. The loss and perplexity by hand
2. A corpus and its context windows
3. A tiny transformer in numpy
4. Pretraining
5. Generating text, before and after
6. The KV cache
7. Scaling in miniature
8. Some numbers: parameters, compute, scaling laws
9. Checks of the exercise sheet
10. What to do next

How to use it: run the cells from top to bottom (Jupyter, VS Code or Google Colab). Only the Python standard library and `numpy` are needed, and no internet connection. Training takes about five minutes on a laptop. The corpus is built from the documentation of your Python installation, so your numbers will differ a little from the outputs saved here: the comments refer to the saved outputs.


## 1. The loss and perplexity by hand

The numbers of the slides. The cross-entropy loss at one position is minus the log probability of the true next token; the loss of a sequence is the mean over its positions; perplexity is the exponent of the mean loss, or equivalently the inverse probability of the sequence normalized by its length.


In [1]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)

# the distribution after "So long and thanks for" (Fig. 7.17 of the book)
for tok, p in [("all", 0.44), ("the", 0.33), ("your", 0.15), ("that", 0.08)]:
    print(f"true next token {tok:5s}  probability {p:.2f}  loss {-np.log(p):.2f}")

# the sequence "So long and thanks for all": probabilities of the true next tokens (invented, except 0.44)
p = np.array([0.20, 0.50, 0.40, 0.80, 0.44])
losses = -np.log(p)
print("\nlosses", losses.round(2), " mean", round(losses.mean(), 4))
print("perplexity from the product:", round(np.prod(p) ** (-1 / len(p)), 4))
print("perplexity from the loss:   ", round(np.exp(losses.mean()), 4))

# the caveat of the tokenizer: the same text, probability 0.001, in 4 or in 6 tokens
print(
    "\ntokenizer A, 4 tokens:",
    round(0.001 ** (-1 / 4), 2),
    "  tokenizer B, 6 tokens:",
    round(0.001 ** (-1 / 6), 2),
)

true next token all    probability 0.44  loss 0.82
true next token the    probability 0.33  loss 1.11
true next token your   probability 0.15  loss 1.90
true next token that   probability 0.08  loss 2.53

losses [1.61 0.69 0.92 0.22 0.82]  mean 0.8526
perplexity from the product: 2.3457
perplexity from the loss:    2.3457

tokenizer A, 4 tokens: 5.62   tokenizer B, 6 tokens: 3.16


## 2. A corpus and its context windows

Our pretraining corpus: the docstrings of about 130 modules of the Python standard library, their classes, functions and methods (a larger version of the corpus of the notebooks of Lectures 04 and 15). Each docstring is a document. We lowercase, keep words only, and use a vocabulary of the 1,498 most frequent words plus two special tokens: `<eot>`, the end-of-text token between documents, and `<unk>` for all the other words. One document in ten is kept apart as validation text, never seen in training.


In [2]:
import collections
import importlib
import inspect
import re
import time
import warnings

warnings.filterwarnings("ignore")

MODULES = [
    "collections",
    "os",
    "re",
    "json",
    "math",
    "random",
    "string",
    "itertools",
    "functools",
    "pathlib",
    "datetime",
    "textwrap",
    "statistics",
    "heapq",
    "bisect",
    "csv",
    "argparse",
    "logging",
    "shutil",
    "subprocess",
    "threading",
    "queue",
    "socket",
    "email",
    "html",
    "http",
    "urllib",
    "unittest",
    "doctest",
    "inspect",
    "typing",
    "dataclasses",
    "enum",
    "decimal",
    "fractions",
    "zipfile",
    "tarfile",
    "gzip",
    "pickle",
    "copy",
    "pprint",
    "difflib",
    "calendar",
    "time",
    "io",
    "tempfile",
    "glob",
    "fnmatch",
    "struct",
    "array",
    "weakref",
    "abc",
    "contextlib",
    "operator",
    "types",
    "asyncio",
    "concurrent.futures",
    "configparser",
    "email.message",
    "http.client",
    "http.server",
    "urllib.request",
    "urllib.parse",
    "xml.dom.minidom",
    "xml.etree.ElementTree",
    "sqlite3",
    "ssl",
    "smtplib",
    "ftplib",
    "imaplib",
    "mailbox",
    "multiprocessing",
    "tokenize",
    "ast",
    "dis",
    "pdb",
    "profile",
    "trace",
    "traceback",
    "warnings",
    "codecs",
    "locale",
    "gettext",
    "shlex",
    "cmd",
    "selectors",
    "signal",
    "zoneinfo",
    "uuid",
    "hashlib",
    "hmac",
    "secrets",
    "base64",
    "binascii",
    "quopri",
    "plistlib",
    "netrc",
    "mimetypes",
    "wave",
    "colorsys",
    "numbers",
    "cmath",
    "graphlib",
    "ipaddress",
    "sched",
    "lzma",
    "bz2",
    "zlib",
    "filecmp",
    "fileinput",
    "linecache",
    "pkgutil",
    "importlib",
    "runpy",
    "sysconfig",
    "platform",
    "optparse",
    "getopt",
]
texts, seen = [], set()


def add(obj):
    d = getattr(obj, "__doc__", None)
    if isinstance(d, str) and d not in seen:  # each docstring once (deduplication!)
        seen.add(d)
        texts.append(d)


for name in MODULES:
    try:
        mod = importlib.import_module(name)
    except Exception:
        continue
    add(mod)
    for _, obj in inspect.getmembers(mod):
        if (inspect.isfunction(obj) or inspect.isclass(obj)) and getattr(
            obj, "__module__", ""
        ) == name:
            add(obj)
            if inspect.isclass(obj):
                for _, meth in inspect.getmembers(obj):
                    if (
                        inspect.isfunction(meth)
                        and getattr(meth, "__module__", "") == name
                    ):
                        add(meth)
docs = [re.findall(r"[a-z]+", t.lower()) for t in texts]
docs = [d for d in docs if len(d) >= 3]
print(len(docs), "documents,", sum(map(len, docs)), "tokens")

counts = collections.Counter(w for d in docs for w in d)
itos = ["<eot>", "<unk>"] + [w for w, _ in counts.most_common(1498)]
stoi = {w: i for i, w in enumerate(itos)}
V = len(itos)

rng = np.random.default_rng(0)
order = rng.permutation(len(docs))
n_val = len(docs) // 10


def pack(indices):
    """all the documents in one stream of token ids, with <eot> after each document"""
    out = []
    for i in indices:
        out += [stoi.get(w, 1) for w in docs[i]] + [0]
    return np.array(out)


val_ids, train_ids = pack(order[:n_val]), pack(order[n_val:])
print(
    "training tokens",
    len(train_ids),
    " validation tokens",
    len(val_ids),
    " vocabulary",
    V,
    f" share of <unk> {np.mean(train_ids == 1):.3f}",
)

2721 documents, 109258 tokens
training tokens 101781  validation tokens 10198  vocabulary 1500  share of <unk> 0.094


In [3]:
N = 32  # the context window of our model, in tokens


def decode(ids):
    return " ".join(itos[i] for i in ids)


print("one training window of", N, "tokens (documents packed, separated by <eot>):\n")
print(decode(train_ids[1000 : 1000 + N]))
print("\nits targets, the same tokens shifted by one:\n")
print(decode(train_ids[1001 : 1001 + N + 0]))

one training window of 32 tokens (documents packed, separated by <eot>):

instance with the specified encoding error handling behavior and line ending s <eot> <unk> a block of data refer to <unk> s docstring for a description of the optional arguments format check

its targets, the same tokens shifted by one:

with the specified encoding error handling behavior and line ending s <eot> <unk> a block of data refer to <unk> s docstring for a description of the optional arguments format check and


Every position of the window is a training example: the input is the window, the targets are the same tokens shifted by one position. When a document ends, `<eot>` separates it from the next one, as in the packing of the lecture.


## 3. A tiny transformer in numpy

One transformer block, as in Lecture 14, with the input and the head of Lecture 15: token embeddings E and learned position embeddings, prenorm layer norm, single-head causal attention, a feedforward layer with ReLU, a final layer norm and the language modeling head with weight tying (logits = h Eᵀ). The loss is the mean cross-entropy over all the positions of all the windows of a batch: teacher forcing, all positions in parallel.

The backward pass is written by hand: it is the error backpropagation of Chapter 6, applied to each operation in reverse order. We check it against numerical gradients before training.


In [4]:
d, d_ff = 64, 256  # model dimension and feedforward dimension (4 d)
MASK = np.triu(np.full((N, N), -1e9), 1)  # the causal mask: no attention to the future


def init_params(d, d_ff, seed=1):
    r = np.random.default_rng(seed)
    s = 0.02
    return {
        "E": r.normal(0, s, (V, d)),
        "P": r.normal(0, s, (N, d)),
        "g1": np.ones(d),
        "b1": np.zeros(d),
        "Wq": r.normal(0, s, (d, d)),
        "Wk": r.normal(0, s, (d, d)),
        "Wv": r.normal(0, s, (d, d)),
        "Wo": r.normal(0, s, (d, d)),
        "g2": np.ones(d),
        "b2": np.zeros(d),
        "W1": r.normal(0, s, (d, d_ff)),
        "c1": np.zeros(d_ff),
        "W2": r.normal(0, s, (d_ff, d)),
        "c2": np.zeros(d),
        "g3": np.ones(d),
        "b3": np.zeros(d),
    }


def softmax(z, axis=-1):
    z = z - z.max(axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis, keepdims=True)


def layer_norm(x, g, b):
    mu = x.mean(-1, keepdims=True)
    sd = np.sqrt(x.var(-1, keepdims=True) + 1e-5)
    xh = (x - mu) / sd
    return xh * g + b, (xh, sd)


def layer_norm_back(dy, cache, g):
    xh, sd = cache
    dxh = dy * g
    dx = (
        dxh - dxh.mean(-1, keepdims=True) - xh * (dxh * xh).mean(-1, keepdims=True)
    ) / sd
    return dx, (dy * xh).sum((0, 1)), dy.sum((0, 1))


def forward(p, ids, targets=None):
    """ids: [B x T] token ids. Returns the logits [B x T x V], or the mean loss if targets are given."""
    B, T = ids.shape
    dd = p["Wq"].shape[0]
    c = {"ids": ids}
    x0 = p["E"][ids] + p["P"][:T]  # token + position embeddings
    n1, c["ln1"] = layer_norm(x0, p["g1"], p["b1"])
    q, k, v = n1 @ p["Wq"], n1 @ p["Wk"], n1 @ p["Wv"]
    A = softmax(
        q @ k.transpose(0, 2, 1) / np.sqrt(dd) + MASK[:T, :T]
    )  # causal attention weights
    o = A @ v
    x1 = x0 + o @ p["Wo"]  # add to the residual stream
    n2, c["ln2"] = layer_norm(x1, p["g2"], p["b2"])
    pre = n2 @ p["W1"] + p["c1"]
    hid = np.maximum(pre, 0)
    x2 = x1 + hid @ p["W2"] + p["c2"]  # feedforward, add to the stream
    n3, c["ln3"] = layer_norm(x2, p["g3"], p["b3"])
    logits = n3 @ p["E"].T  # language modeling head, weight tying
    c.update(n1=n1, q=q, k=k, v=v, A=A, o=o, n2=n2, pre=pre, hid=hid, n3=n3)
    if targets is None:
        return logits
    prob = softmax(logits)
    loss = -np.log(
        np.take_along_axis(prob, targets[..., None], -1)
    ).mean()  # mean cross-entropy
    c.update(prob=prob, targets=targets)
    return loss, c


def backward(p, c):
    g = {k: np.zeros_like(v) for k, v in p.items()}
    B, T = c["ids"].shape
    dd = p["Wq"].shape[0]
    dl = c["prob"].copy()  # d loss / d logits = (y_hat - y) / (B T)
    np.put_along_axis(
        dl,
        c["targets"][..., None],
        np.take_along_axis(dl, c["targets"][..., None], -1) - 1,
        -1,
    )
    dl /= B * T
    g["E"] += np.einsum("btv,btd->vd", dl, c["n3"])
    dn3 = dl @ p["E"]
    dx2, g["g3"], g["b3"] = layer_norm_back(dn3, c["ln3"], p["g3"])
    dx1 = dx2.copy()
    g["c2"] = dx2.sum((0, 1))
    g["W2"] = np.einsum("bti,btj->ij", c["hid"], dx2)
    dpre = (dx2 @ p["W2"].T) * (c["pre"] > 0)
    g["c1"] = dpre.sum((0, 1))
    g["W1"] = np.einsum("bti,btj->ij", c["n2"], dpre)
    d_, g["g2"], g["b2"] = layer_norm_back(dpre @ p["W1"].T, c["ln2"], p["g2"])
    dx1 += d_
    dx0 = dx1.copy()
    g["Wo"] = np.einsum("bti,btj->ij", c["o"], dx1)
    do = dx1 @ p["Wo"].T
    A = c["A"]
    dA = do @ c["v"].transpose(0, 2, 1)
    dv = A.transpose(0, 2, 1) @ do
    dS = A * (dA - (dA * A).sum(-1, keepdims=True)) / np.sqrt(dd)
    dq = dS @ c["k"]
    dk = dS.transpose(0, 2, 1) @ c["q"]
    for name, dz in (("Wq", dq), ("Wk", dk), ("Wv", dv)):
        g[name] = np.einsum("bti,btj->ij", c["n1"], dz)
    dn1 = dq @ p["Wq"].T + dk @ p["Wk"].T + dv @ p["Wv"].T
    d_, g["g1"], g["b1"] = layer_norm_back(dn1, c["ln1"], p["g1"])
    dx0 += d_
    np.add.at(g["E"], c["ids"], dx0)
    g["P"][:T] += dx0.sum(0)  # the embeddings are trained too
    return g


params = init_params(d, d_ff)
n_par = sum(v.size for v in params.values())
print(
    f"parameters: {n_par:,}  (E alone: {params['E'].size:,}; one block, without layer norms: {12 * d * d + d_ff + d:,})"
)

parameters: 147,904  (E alone: 96,000; one block, without layer norms: 49,472)


The count of the lecture, 12 d² for a block (here 49,152, plus the biases of the feedforward layer), and |V| d for E. With such a small d, unlike GPT-3, the embeddings are most of the parameters.

**Gradient check.** For a few weights, we compare the gradient of the backward pass with a numerical estimate, (loss(w + ε) − loss(w − ε)) / 2ε.


In [5]:
x = train_ids[:18].reshape(2, 9)
y = train_ids[1:19].reshape(2, 9)
loss, cache = forward(params, x, y)
grads = backward(params, cache)
r = np.random.default_rng(3)
for name in ["E", "P", "Wq", "Wk", "Wv", "Wo", "W1", "W2", "g1", "b2"]:
    idx = (
        (x[0, 3], 5)
        if name == "E"
        else (2, 5)
        if name == "P"
        else tuple(r.integers(0, s) for s in params[name].shape)
    )
    old = params[name][idx]
    eps = 1e-5
    params[name][idx] = old + eps
    lp, _ = forward(params, x, y)
    params[name][idx] = old - eps
    lm, _ = forward(params, x, y)
    params[name][idx] = old
    print(
        f"{name:3s}  backward {grads[name][idx]: .8f}   numerical {(lp - lm) / (2 * eps): .8f}"
    )
print(f"\nloss before training: {loss:.3f}   ln |V| = {np.log(V):.3f}")

E    backward -0.00906825   numerical -0.00906825
P    backward  0.03175116   numerical  0.03175116
Wq   backward  0.00021442   numerical  0.00021442
Wk   backward  0.00045641   numerical  0.00045641
Wv   backward  0.00703784   numerical  0.00703784
Wo   backward  0.00064865   numerical  0.00064865
W1   backward -0.01441500   numerical -0.01441500
W2   backward  0.01613121   numerical  0.01613121
g1   backward -0.00108378   numerical -0.00108378
b2   backward  0.00108205   numerical  0.00108205

loss before training: 7.303   ln |V| = 7.313


The two columns agree. Before training the model gives almost the same probability to every token, so its loss is close to ln |V| (exercise 1c of the sheet).


## 4. Pretraining

Training: random windows of N = 32 tokens from the training stream, 16 windows per batch (512 predictions per step), the mean cross-entropy loss, gradient descent with the Adam optimizer and a learning rate that decreases along training. Every 300 steps we compute the mean loss and the perplexity on the validation text. For reference, two models of Lecture 04 on the same tokens: a unigram model and a bigram model (interpolated with the unigram, so that no probability is 0).


In [6]:
def windows(data):
    n = (len(data) - 1) // N
    return data[: n * N].reshape(n, N), data[1 : n * N + 1].reshape(n, N)


def evaluate(p, data):
    xs, ys = windows(data)
    total = 0.0
    for i in range(0, len(xs), 64):
        l, _ = forward(p, xs[i : i + 64], ys[i : i + 64])
        total += l * len(xs[i : i + 64])
    return total / len(xs)


# n-gram baselines on the same tokens (Lecture 04)
uni = np.bincount(train_ids, minlength=V) + 1.0
uni /= uni.sum()
big = np.zeros((V, V))
np.add.at(big, (train_ids[:-1], train_ids[1:]), 1)
big_p = big / np.maximum(big.sum(1, keepdims=True), 1)
xs, ys = windows(val_ids)
xs, ys = xs.ravel(), ys.ravel()
print(
    f"unigram   validation loss {-np.mean(np.log(uni[ys])):.3f}  perplexity {np.exp(-np.mean(np.log(uni[ys]))):.1f}"
)
lp = -np.mean(np.log(0.7 * big_p[xs, ys] + 0.3 * uni[ys]))
print(f"bigram    validation loss {lp:.3f}  perplexity {np.exp(lp):.1f}")


def train(p, steps, batch=16, lr=3e-3, seed=0, log_every=300, quiet=False):
    m = {k: np.zeros_like(v) for k, v in p.items()}
    s = {k: np.zeros_like(v) for k, v in p.items()}
    r = np.random.default_rng(seed)
    b1, b2 = 0.9, 0.99
    history = []
    for step in range(1, steps + 1):
        lr_t = lr * (
            0.1 + 0.45 * (1 + np.cos(np.pi * step / steps))
        )  # from lr down to lr / 10
        starts = r.integers(0, len(train_ids) - N - 1, batch)
        x = np.stack([train_ids[a : a + N] for a in starts])
        y = np.stack([train_ids[a + 1 : a + N + 1] for a in starts])
        loss, cache = forward(p, x, y)
        g = backward(p, cache)  # teacher forcing: inputs are the true tokens
        for k in p:  # Adam
            m[k] = b1 * m[k] + (1 - b1) * g[k]
            s[k] = b2 * s[k] + (1 - b2) * g[k] ** 2
            p[k] -= (
                lr_t * (m[k] / (1 - b1**step)) / (np.sqrt(s[k] / (1 - b2**step)) + 1e-8)
            )
        if step % log_every == 0 or step == 1:
            vl = evaluate(p, val_ids)
            history.append((step, loss, vl))
            if not quiet:
                print(
                    f"step {step:5d}  batch loss {loss:.3f}  validation loss {vl:.3f}  perplexity {np.exp(vl):7.1f}"
                )
    return history


params = init_params(d, d_ff)
before = {k: v.copy() for k, v in params.items()}
t0 = time.time()
history = train(params, 3000)
print(f"\n{time.time() - t0:.0f} seconds, {3000 * 16 * N:,} training predictions")
print(
    f"final validation perplexity: transformer {np.exp(history[-1][2]):.1f}, bigram {np.exp(lp):.1f}"
)

unigram   validation loss 5.734  perplexity 309.1
bigram    validation loss 4.674  perplexity 107.1
step     1  batch loss 7.328  validation loss 7.181  perplexity  1314.8
step   300  batch loss 5.449  validation loss 5.521  perplexity   250.0
step   600  batch loss 5.058  validation loss 5.092  perplexity   162.7
step   900  batch loss 4.567  validation loss 4.907  perplexity   135.2
step  1200  batch loss 4.538  validation loss 4.757  perplexity   116.4
step  1500  batch loss 4.337  validation loss 4.691  perplexity   108.9
step  1800  batch loss 4.150  validation loss 4.618  perplexity   101.3
step  2100  batch loss 4.003  validation loss 4.585  perplexity    98.0
step  2400  batch loss 3.999  validation loss 4.552  perplexity    94.8
step  2700  batch loss 4.038  validation loss 4.543  perplexity    94.0
step  3000  batch loss 3.899  validation loss 4.531  perplexity    92.8

162 seconds, 1,536,000 training predictions
final validation perplexity: transformer 92.8, bigram 107.1


The validation loss starts at ln |V| ≈ 7.3 (perplexity about |V|) and goes down, fast at first and then slowly: the model has learned something general about the docstrings, since the validation text was never seen. Compare it with the bigram model: on so little text, about 100,000 tokens, counting bigrams is a strong baseline, and our tiny transformer, trained for two minutes, does not beat it (in the saved run, a perplexity of about 150 against 125). Transformers pull away with much more data, context and compute: this is what the scaling laws of the lecture are about. A transformer trained on trillions of tokens goes much further, but the procedure is the same: predict the next token, compute the cross-entropy, backpropagate, repeat.

The batch loss is lower than the validation loss: the model fits its training text better than new text. With much more data, this gap would shrink.


## 5. Generating text, before and after

Generation with the decoding of Lecture 15: temperature 0.8 and top-k with k = 20, from the same prompt, with the weights before training and after.


In [7]:
def generate(p, prompt, n=20, tau=0.8, top_k=20, seed=0):
    r = np.random.default_rng(seed)
    ids = [stoi.get(w, 1) for w in prompt.split()]
    for _ in range(n):
        ctx = np.array(ids[-N:])[None]
        logits = forward(p, ctx)[0, -1] / tau
        keep = np.argsort(logits)[-top_k:]
        probs = softmax(logits[keep])
        ids.append(int(keep[r.choice(len(keep), p=probs)]))
    return decode(ids)


for label, p in (("before training", before), ("after training ", params)):
    for seed in (0, 1):
        print(f"{label}: {generate(p, 'return the', seed=seed)}")

before training: return the oserror do want chain token source which report tell tell st inside occurred actions which future multipart instead pairs without
before training: return the specific produced has has typed none expected modified buffer off esmtp item defaults multipart st associated off token print yet
after training : return the object the first module s module name of the module s also <unk> a module constructor <eot> <unk> a new
after training : return the value of the value in the value is a bytes object to a string in a single sequence and any


Before training: random words. After training: not good English, but the style and many local patterns of the docstrings (_return the ... of the ..._, _if ... is not ..._). The `<eot>` token can also be generated: the model has learned that documents end.


## 6. The KV cache

At inference we generate one token at a time. Without a cache, at each step we run the model on the whole context again, recomputing the keys and values of all the previous tokens. With the KV cache we compute the query, key and value of the new token only, store the key and the value, and attend over the stored ones. The two give the same logits.


In [8]:
def step_cached(p, token, pos, cache):
    """one new token: its query, key and value; the keys and values of the previous tokens come from the cache"""
    x0 = p["E"][token] + p["P"][pos]
    n1, _ = layer_norm(x0[None, None], p["g1"], p["b1"])
    n1 = n1[0, 0]
    q, k, v = n1 @ p["Wq"], n1 @ p["Wk"], n1 @ p["Wv"]
    cache["K"].append(k)
    cache["V"].append(v)  # store
    K, Vv = np.array(cache["K"]), np.array(cache["V"])
    a = softmax(K @ q / np.sqrt(len(q)))
    x1 = x0 + (a @ Vv) @ p["Wo"]
    n2, _ = layer_norm(x1[None, None], p["g2"], p["b2"])
    n2 = n2[0, 0]
    x2 = x1 + np.maximum(n2 @ p["W1"] + p["c1"], 0) @ p["W2"] + p["c2"]
    n3, _ = layer_norm(x2[None, None], p["g3"], p["b3"])
    return n3[0, 0] @ p["E"].T


ids = train_ids[2000 : 2000 + N]
cache = {"K": [], "V": []}
cached = np.array([step_cached(params, t, i, cache) for i, t in enumerate(ids)])
full = forward(params, ids[None])[0]
print(
    "largest difference between the logits with and without the cache:",
    np.abs(cached - full).max(),
)

T = N
print(
    f"\nkey and value computations to process {T} tokens one at a time: without the cache {T * (T + 1) // 2}, with the cache {T}"
)
t0 = time.time()
for t in range(1, T + 1):
    forward(params, ids[None, :t])
t_full = time.time() - t0
t0 = time.time()
cache = {"K": [], "V": []}
for i, t in enumerate(ids):
    step_cached(params, t, i, cache)
t_cache = time.time() - t0
print(
    f"time without the cache {1000 * t_full:.1f} ms, with the cache {1000 * t_cache:.1f} ms (it depends on your computer)"
)
print(f"memory of the cache: 2 x {T} x {d} = {2 * T * d} numbers, one layer")

largest difference between the logits with and without the cache: 1.0658141036401503e-14

key and value computations to process 32 tokens one at a time: without the cache 528, with the cache 32
time without the cache 15.3 ms, with the cache 11.4 ms (it depends on your computer)
memory of the cache: 2 x 32 x 64 = 4096 numbers, one layer


The logits are the same up to rounding errors. The saving grows with the length of the context: for 1,000 tokens, 500,500 key and value computations per layer against 1,000 (exercise 7c). The price is memory, which grows with the context: for the large models of Fig. 7.24, gigabytes per sequence (section 8).


## 7. Scaling in miniature

The scaling laws of Kaplan et al. (2020) relate the loss to the number of parameters outside the embeddings, when data and compute do not limit it. Here three models with d = 16, 32 and 64 (one block each), trained for the same 800 steps on the same data. We measure the loss on 200 fixed windows of the training text and on the validation text.


In [9]:
results = []
for dd in (16, 32, 64):
    p = init_params(dd, 4 * dd, seed=2)
    h = train(p, 800, log_every=800, quiet=True)
    non_emb = sum(v.size for k, v in p.items() if k not in ("E", "P"))
    tr = evaluate(p, train_ids[: 200 * N + 1])
    results.append((dd, non_emb, tr, h[-1][2]))
    print(
        f"d = {dd:3d}   parameters outside the embeddings {non_emb:7,}   training loss {tr:.3f}   validation loss {h[-1][2]:.3f}"
    )

d =  16   parameters outside the embeddings   3,248   training loss 5.127   validation loss 5.224
d =  32   parameters outside the embeddings  12,640   training loss 4.933   validation loss 5.071
d =  64   parameters outside the embeddings  49,856   training loss 4.792   validation loss 4.970


The largest model, d = 64, has the lowest loss, on the training text and on the validation text. Between d = 16 and d = 32 the order can go either way: with tiny models, a few hundred steps and 100,000 tokens, the differences are within the noise of the random initialization and of the data (your Python version gives a slightly different corpus; try other seeds). Scaling laws describe a trend across many orders of magnitude, for models trained on enough data: Kaplan et al. fitted L(N) = (N / 8.8 × 10¹³)^(−0.076) on models from about 10³ to 10⁹ parameters, where data and compute do not limit the loss. A factor of 4 in our tiny models is not enough to see it reliably.


## 8. Some numbers: parameters, compute, scaling laws

The models of Fig. 7.24 with the count of the lecture, 12 L d² + |V| d (exact for GPT-2 and GPT-3, rough for the newer models, which use GQA and SwiGLU), and the KV cache with 16-bit numbers and keys and values of size d. Then the training compute 6 N D and the scaling laws.


In [10]:
models = [
    ("GPT-2 XL", 48, 1600, 50257, 1024, 1.5e9),
    ("GPT-3", 96, 12288, 50257, 2048, 175e9),
    ("Llama 3.1 8B", 32, 4096, 128256, 131072, 8e9),
    ("Llama 3.1 405B", 126, 16384, 128256, 131072, 405e9),
]
print(
    f"{'model':15s} {'12 L d^2 + |V| d':>18s} {'table':>8s} {'KV cache per token':>20s} {'full context':>14s}"
)
for name, L, dm, Vm, ctx, table in models:
    count = 12 * L * dm**2 + Vm * dm
    kv = 2 * L * dm * 2
    print(
        f"{name:15s} {count / 1e9:16.2f}B {table / 1e9:7.1f}B {kv / 1e6:17.2f} MB {kv * ctx / 1e9:11.1f} GB"
    )
print(
    "\nthe rough count is far from the table for Llama 3.1 8B: SwiGLU uses 3 matrices with d_ff = 3.5 d, and GQA smaller keys and values"
)

print("\ntraining compute, 6 N D:")
for name, Np, D in (
    ("GPT-3", 175e9, 300e9),
    ("Gopher", 280e9, 300e9),
    ("Chinchilla", 70e9, 1.4e12),
    ("Llama 3.1 8B", 8e9, 15e12),
    ("Llama 3.1 405B", 405e9, 15e12),
):
    print(f"  {name:15s} {6 * Np * D:9.2e} FLOPs   {D / Np:7.1f} tokens per parameter")

L_N = lambda n: (n / 8.8e13) ** -0.076
print("\nKaplan et al. (2020), L(N) = (N / 8.8e13)^-0.076:")
for n in (1e8, 1e9, 1e10):
    print(f"  N = {n:.0e}   loss {L_N(n):.3f} nats   perplexity {np.exp(L_N(n)):.1f}")
print(f"  2 times larger: x {2**-0.076:.3f}   10 times larger: x {10**-0.076:.3f}")

model             12 L d^2 + |V| d    table   KV cache per token   full context
GPT-2 XL                    1.55B     1.5B              0.31 MB         0.3 GB
GPT-3                     174.56B   175.0B              4.72 MB         9.7 GB
Llama 3.1 8B                6.97B     8.0B              0.52 MB        68.7 GB
Llama 3.1 405B            407.98B   405.0B              8.26 MB      1082.3 GB

the rough count is far from the table for Llama 3.1 8B: SwiGLU uses 3 matrices with d_ff = 3.5 d, and GQA smaller keys and values

training compute, 6 N D:
  GPT-3            3.15e+23 FLOPs       1.7 tokens per parameter
  Gopher           5.04e+23 FLOPs       1.1 tokens per parameter
  Chinchilla       5.88e+23 FLOPs      20.0 tokens per parameter
  Llama 3.1 8B     7.20e+23 FLOPs    1875.0 tokens per parameter
  Llama 3.1 405B   3.65e+25 FLOPs      37.0 tokens per parameter

Kaplan et al. (2020), L(N) = (N / 8.8e13)^-0.076:
  N = 1e+08   loss 2.830 nats   perplexity 16.9
  N = 1e+09   loss 2.37

## 9. Checks of the exercise sheet

The numbers of the solutions of exercises 1 to 3 and 5 to 12 (exercise 4 asks for prompts: there is nothing to compute).


In [11]:
ln = np.log
print(
    "1  ",
    round(-ln(0.6), 3),
    round(-ln(0.1), 3),
    " uniform:",
    round(ln(50000), 2),
    round(ln(1500), 2),
)
q = np.array([0.10, 0.30, 0.50, 0.60, 0.40])
print(
    "2  ",
    (-ln(q)).round(3),
    " sum",
    round(-ln(q).sum(), 3),
    " mean",
    round(-ln(q).mean(), 3),
)
print(
    "3  ",
    3.2e6 / 2048,
    "windows,",
    300e9 / 3.2e6,
    "steps;",
    2048 - (700 + 1 + 900 + 1),
    "tokens fit,",
    600 - (2048 - 1602),
    "go on",
)
a, o = 50000, 950000
print(
    "5  ",
    a * 0.1,
    o * 0.02,
    "removed; share after",
    round(a * 0.9 / (a * 0.9 + o * 0.98), 4),
)
print(
    "6  ",
    12 * 12 * 768**2,
    50257 * 768,
    1024 * 768,
    "total",
    12 * 12 * 768**2 + 50257 * 768 + 1024 * 768,
    "; GPT-3 share of E",
    round(50257 * 12288 / (12 * 96 * 12288**2 + 50257 * 12288), 4),
)
print(
    "7  ",
    2 * 48 * 1600,
    "numbers,",
    2 * 48 * 1600 * 2,
    "bytes,",
    round(2 * 48 * 1600 * 2 * 1024 / 1e6, 1),
    "MB;",
    round(2 * 126 * 16384 * 2 * 131072 / 1e12, 2),
    "TB,",
    round(2 * 126 * 1024 * 2 * 131072 / 1e9, 1),
    "GB;",
    sum(range(1, 1001)),
)
print(
    "8  ",
    round(L_N(1e8), 2),
    round(np.exp(L_N(1e8)), 1),
    round(L_N(1e9), 2),
    round(np.exp(L_N(1e9)), 1),
    "; halve:",
    round(2 ** (1 / 0.076)),
    "; 10x data:",
    round(10**-0.095, 3),
)
C = 6 * 70e9 * 1.4e12
Nopt = np.sqrt(C / 120)
print(
    "9  ",
    f"{C:.3g}",
    f"N {Nopt:.3g} D {20 * Nopt:.3g}",
    f"4C: N {np.sqrt(4 * C / 120):.3g} D {20 * np.sqrt(4 * C / 120):.3g}",
    f"Llama 8B {6 * 8e9 * 15e12:.3g} ratio {6 * 8e9 * 15e12 / C:.2f}",
)
print(
    "10 ",
    round(np.prod(q) ** (-1 / 5), 3),
    round(np.exp(-ln(q).mean()), 3),
    "; 2.38 nats:",
    round(np.exp(2.38), 1),
    round(2.38 / ln(2), 2),
    "bits,",
    round(2 ** (2.38 / ln(2)), 1),
)
print(
    "11 ",
    round(1e-6 ** (-1 / 3), 1),
    round(1e-6 ** (-1 / 5), 1),
    round(1e-6 ** (-1 / 2), 1),
)


def overlap(out, ref, clip=True):
    ref_counts = collections.Counter(ref.split())
    n = 0
    for w in out.split():
        if ref_counts[w] > 0:
            n += 1
            if clip:
                ref_counts[w] -= 1
    return n


ref = "the president met the press on monday"
print(
    "12 ",
    (800 - 190) / 800,
    overlap("on monday the president met reporters", ref),
    overlap("the press the press the press", ref),
    overlap("the press the press the press", ref, clip=False),
)

1   0.511 2.303  uniform: 10.82 7.31
2   [2.303 1.204 0.693 0.511 0.916]  sum 5.627  mean 1.125
3   1562.5 windows, 93750.0 steps; 446 tokens fit, 154 go on
5   5000.0 19000.0 removed; share after 0.0461
6   84934656 38597376 786432 total 124318464 ; GPT-3 share of E 0.0035
7   153600 numbers, 307200 bytes, 314.6 MB; 1.08 TB, 67.6 GB; 500500
8   2.83 16.9 2.38 10.8 ; halve: 9139 ; 10x data: 0.804
9   5.88e+23 N 7e+10 D 1.4e+12 4C: N 1.4e+11 D 2.8e+12 Llama 8B 7.2e+23 ratio 1.22
10  3.081 3.081 ; 2.38 nats: 10.8 3.43 bits, 10.8
11  100.0 15.8 1000.0
12  0.7625 5 3 6


## 10. What to do next

- Train longer, or with a larger `d`: can you beat the bigram model on the validation text? Watch the gap between batch loss and validation loss.
- Change `d`, or add a second block (stack two blocks in `forward` and `backward`): does the validation loss go down?
- Remove the causal mask and look at the training loss: why does it collapse, and why would such a model be useless for generation?
- Try d = 128 with 800 steps in section 7: is it better than d = 64? With a fixed budget of steps, a larger model is not always better (the point of Hoffmann et al., 2022).
